# Sommelier Kaggle Full Trace Run - MossFormer2 Stage 3 + PhoWhisper Local

Notebook này là bản single-file để chạy full pipeline trên Kaggle:
- Tự clone repo.
- Tự cài dependencies từ internet.
- Stage 3 dùng `ClearVoice` + `MossFormer2_SS_16K` self-host/local để tách speech overlap, không gọi Hugging Face Inference API.
- Stage 4 chạy Whisper large-v3 + PhoWhisper local + ChunkFormer local, không dùng PhoWhisper API.
- Không phụ thuộc notebook `00_v00_build_wheels_dataset.ipynb` hay private wheels dataset.

Điều kiện trước khi chạy:
- Kaggle Accelerator: GPU bật cho các stage nặng.
- Kaggle Internet: bật để cài dependencies và tải model local lần đầu.
- Kaggle Secret có `HF_TOKEN` cho các model gated như pyannote.
- Dataset audio đã Add Input vào notebook.

Lưu ý: bản này tối ưu cho podcast. Stage 3 mark overlap từ 0.05s và thử MossFormer từ 0.10s để giữ các backchannel ngắn như “vâng”, “ừ”, “dạ”, “đúng rồi”. Các đoạn micro overlap vẫn được gắn nhãn review để không đưa nhầm vào clean train.


## 0. Cấu hình run

Chỉnh các biến bên dưới nếu muốn đổi branch, giới hạn thời lượng test, hoặc tắt bước nặng.

In [ ]:
REPO_URL = "https://github.com/lamkdhe180931-arch/sommelier.git"
BRANCH = "test-divide-stage"

RUN_DIR = "/kaggle/working/run_full"
INPUT_DIR = f"{RUN_DIR}/00_input"
DIAR_DIR = f"{RUN_DIR}/01_diarization"
MUSIC_DIR = f"{RUN_DIR}/02_music_clean"
OVERLAP_DIR = f"{RUN_DIR}/03_overlap"
ASR_DIR = f"{RUN_DIR}/04_asr"
EXPORT_DIR = f"{RUN_DIR}/05_export"
FINAL_DIR = f"{EXPORT_DIR}/final"
EVAL_DIR = f"{RUN_DIR}/06_eval"
PREVIEW_DIR = f"{RUN_DIR}/preview"
LOG_DIR_PATH = f"{RUN_DIR}/logs"
AUDIO_WAV = f"{INPUT_DIR}/full.wav"

# Để None nếu muốn chạy full audio. Để 300 nếu muốn test nhanh 5 phút.
AUDIO_LIMIT_SECONDS = 300

# Bước nặng. Có thể tắt để debug nhanh.
RUN_DEMUCS = True

# Stage 01 podcast cleanup: chỉ merge cùng speaker khi gap rất nhỏ, giữ backchannel khác speaker.
SAME_SPEAKER_MERGE_GAP_SECONDS = 0.30
SHORT_BACKCHANNEL_SECONDS = 1.0

# Full-duplex train grouping: chọn 2 speaker chính rồi chia clean/review/exclude.
EXPECTED_MAIN_SPEAKERS = 2
MAIN_SPEAKERS = []  # Ví dụ ["SPEAKER_04", "SPEAKER_05"] nếu muốn ép 2 người chính thủ công.
EXPORT_PARTITION_BY_DUPLEX_GROUP = True

# Stage 03 dùng ClearVoice/MossFormer2 self-host để tách overlap, không gọi API.
RUN_OVERLAP_MARK_ONLY = False
RUN_MOSSFORMER_SEPARATION = True
MOSSFORMER_MODEL_NAME = "MossFormer2_SS_16K"
MOSSFORMER_TASK = "speech_separation"
MOSSFORMER_FAIL_OPEN = True
MOSSFORMER_MIN_OVERLAP_SECONDS = 0.10
MOSSFORMER_MIN_SEGMENT_SECONDS = 0.15
MOSSFORMER_CONTEXT_SECONDS = 0.8
MOSSFORMER_MAX_WINDOW_SECONDS = 6.0
MOSSFORMER_SAVE_WINDOW_STEMS = True

# Speaker-safe stem assignment: dùng embedding để map stem -> speaker; low-confidence không đưa sang ASR.
MOSSFORMER_USE_SPEAKER_EMBEDDING_ASSIGNMENT = True
MOSSFORMER_EMBEDDING_DEVICE = "cpu"  # CPU tránh chiếm thêm VRAM khi ClearVoice đang chạy GPU.
MOSSFORMER_EMBEDDING_FAIL_OPEN = True
MOSSFORMER_REFERENCE_MIN_SECONDS = 1.5
MOSSFORMER_REFERENCE_MAX_SEGMENTS_PER_SPEAKER = 6
MOSSFORMER_ASSIGNMENT_MIN_CONFIDENCE = 0.55
MOSSFORMER_ASSIGNMENT_MIN_MARGIN = 0.08
MOSSFORMER_REQUIRE_EMBEDDING_ASSIGNMENT = True
MOSSFORMER_DISABLE_LOW_CONFIDENCE_ASR = True

OVERLAP_MARK_THRESHOLD_SECONDS = 0.05
OVERLAP_REVIEW_THRESHOLD_SECONDS = 0.05
OVERLAP_REQUIRE_DIFFERENT_SPEAKER = True
RUN_PYANNOTE_OSD = True
PYANNOTE_OSD_MODEL = "pyannote/overlapped-speech-detection"
PYANNOTE_OSD_MIN_DURATION_SECONDS = 0.05
PYANNOTE_OSD_FAIL_OPEN = True

# Giữ các biến cũ để cell/log không bị nhầm, nhưng stage 03 không dùng SepReformer nữa.
MIN_SEPREFORMER_OVERLAP_SECONDS = 1.0
MIN_SEPREFORMER_SEGMENT_SECONDS = 1.0

# Guard cho ASR ensemble: sửa các đoạn quá ngắn bị Whisper hallucinate.
ASR_QUALITY_GUARD = True
ASR_MICRO_SEGMENT_SECONDS = 0.5
ASR_SHORT_SEGMENT_SECONDS = 1.0
ASR_VI_AGREEMENT_THRESHOLD = 0.75

# ASR context padding: model nghe thêm biên trước/sau nhưng timestamp export vẫn giữ segment gốc.
ASR_CONTEXT_PAD_BEFORE_SECONDS = 0.25
ASR_CONTEXT_PAD_AFTER_SECONDS = 0.35

# ASRMoE chạy cả 3 model tiếng Việt: Whisper + PhoWhisper + ChunkFormer.
# Trên Kaggle 2xT4: Whisper đặt GPU0, PhoWhisper/ChunkFormer đặt GPU1.
ASR_MOE = True
WHISPER_DEVICE_INDEX = 0
VI_ASR_DEVICE_INDEX = 1
WHISPER_ARCH = "large-v3"
COMPUTE_TYPE = "float16"
ASR_THREADS = 4

# Initial prompt chỉ áp dụng cho Whisper. Giữ ngắn để giảm bias/hallucination.
USE_WHISPER_INITIAL_PROMPT = True
WHISPER_INITIAL_PROMPT = (
    "Podcast tiếng Việt tự nhiên, có thể xen từ tiếng Anh, tên app, brand và từ lóng. "
    "Từ đệm/backchannel thường gặp: ừ, ờ, à, dạ, vâng, đúng rồi, rồi, thì, là. "
    "Từ mượn/tên riêng: Facebook, Zalo, YouTube, TikTok, Instagram, Google, AI, livestream, podcast, content, deadline, feedback, booking, trend, viral, team, meeting, plank."
)

# PhoWhisper local/self-host: tải model về Kaggle và chạy local, không gọi HF Inference API.
WHISPER_HOTWORDS = "Facebook, Zalo, YouTube, TikTok, Instagram, Google, AI, livestream, podcast, content, deadline, feedback, booking, trend, viral, team, meeting, plank, vâng, dạ, đúng rồi"

PHOWHISPER_USE_HF_API = False
PHOWHISPER_API_MODEL = "vinai/PhoWhisper-large"
PHOWHISPER_API_PROVIDER = "hf-inference"
PHOWHISPER_API_TIMEOUT_SECONDS = 180

HF_SECRET_NAME = "HF_TOKEN"


In [ ]:

from pathlib import Path
import os
import shlex
import subprocess

LOG_DIR = Path(LOG_DIR_PATH)
for _dir in [INPUT_DIR, DIAR_DIR, MUSIC_DIR, OVERLAP_DIR, ASR_DIR, EXPORT_DIR, FINAL_DIR, EVAL_DIR, PREVIEW_DIR, LOG_DIR_PATH]:
    Path(_dir).mkdir(parents=True, exist_ok=True)


def _format_cmd(cmd):
    if isinstance(cmd, (list, tuple)):
        return " ".join(shlex.quote(str(part)) for part in cmd)
    return str(cmd)


def tail_file(path, n=30):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])


def run_logged(cmd, log_name, cwd=None, env=None, shell=False, tail=20):
    log_path = LOG_DIR / log_name
    cwd = cwd or os.getcwd()
    print("Running:", _format_cmd(cmd))
    print("Log:", log_path)
    with open(log_path, "w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.run(
            cmd,
            cwd=cwd,
            env=env,
            shell=shell,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    print("Exit code:", proc.returncode)
    if tail:
        log_tail = tail_file(log_path, n=tail)
        if log_tail:
            print(f"--- last {tail} log lines ---")
            print(log_tail)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return log_path


def export_audio_preview(audio_segment, out_path, seconds=30):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    audio_segment[: int(seconds * 1000)].export(out_path, format="wav")
    return out_path


## 1. Clone repo

In [ ]:

import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
repo_dir = Path("/kaggle/working/sommelier")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

run_logged(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], "01_clone_repo.log", cwd="/kaggle/working", tail=20)
os.chdir(repo_dir / "podcast-pipeline")
print("cwd:", os.getcwd())
print("branch:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("commit:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())


## 2. Cài dependencies từ internet

Bản standalone cài trực tiếp trong notebook này. Không cần private wheels dataset.


In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

run_logged(["apt-get", "update", "-y"], "02_apt_update.log", tail=10)
run_logged(["apt-get", "install", "-y", "ffmpeg", "git", "git-lfs"], "03_apt_install.log", tail=10)
run_logged(["python", "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "packaging", "ninja"], "04_pip_base.log", tail=12)

req = Path("requirements.txt").read_text(encoding="utf-8")
filtered = [line for line in req.splitlines() if "nemo-toolkit[all]" not in line]
Path("requirements-kaggle.txt").write_text("\n".join(filtered) + "\n", encoding="utf-8")
run_logged(["python", "-m", "pip", "install", "-r", "requirements-kaggle.txt"], "05_pip_requirements.log", tail=20)

run_logged(["python", "-m", "pip", "uninstall", "-y", "nemo-toolkit", "lightning", "pytorch-lightning"], "06_pip_uninstall_nemo.log", tail=8)
run_logged(["python", "-m", "pip", "install", "lightning==2.4.0", "pytorch-lightning==2.5.2"], "07_pip_lightning.log", tail=12)
run_logged(["python", "-m", "pip", "install", "nemo-toolkit[asr]==2.4.0"], "08_pip_nemo_asr.log", tail=20)

run_logged(["python", "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], "09_pip_uninstall_torch.log", tail=8)
run_logged([
    "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
    "torch==2.7.1", "torchaudio==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu126",
], "10_pip_torch_stack.log", tail=20)

run_logged(["python", "-m", "pip", "install", "pillow<12.0"], "11_pip_pillow.log", tail=8)
run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", "torchmetrics==1.7.4"], "12_pip_torchmetrics.log", tail=8)
run_logged([
    "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
    "numpy==2.2.6", "numba==0.61.2", "llvmlite==0.44.0",
], "13_pip_numpy_numba.log", tail=12)

run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "clearvoice==0.1.2"], "14_pip_clearvoice.log", tail=20)

print("Dependency install logs saved in:", LOG_DIR)


## 3. Kiểm tra môi trường

In [ ]:
import importlib.metadata as importlib_metadata
import shutil
import subprocess
import numpy, numba, torch

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; kiểm tra torch.cuda bên dưới.")

print("nemo-toolkit:", importlib_metadata.version("nemo-toolkit"))
print("chunkformer:", importlib_metadata.version("chunkformer"))
print("numpy:", numpy.__version__)
print("numba:", numba.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print("GPU", i, torch.cuda.get_device_name(i))

import whisperx
print("whisperx ok")

import nemo.collections.asr as nemo_asr
print("nemo asr ok")

from nemo.collections.asr.models import SortformerEncLabelModel
print("sortformer import ok")


## 4. Gắn Hugging Face token vào config

In [ ]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

token = UserSecretsClient().get_secret(HF_SECRET_NAME)
print("HF token:", token[:8] + "..." if token else "missing")
print(whoami(token=token))

with open("config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["huggingface_token"] = token

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("config.json updated")

## 5. Tìm audio input và chuẩn hóa audio

Output: `AUDIO_WAV = /kaggle/working/audio/full.wav`.

Chuẩn hóa về mono 16 kHz để các model dùng cùng format.

In [ ]:

from pathlib import Path

audio_exts = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
audio_candidates = sorted(
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in audio_exts
)

if not audio_candidates:
    raise FileNotFoundError("Không tìm thấy audio trong /kaggle/input. Hãy Add Input hoặc Upload audio trước.")

AUDIO_IN = str(audio_candidates[0])
print("AUDIO_IN:", AUDIO_IN)
print("AUDIO_WAV:", AUDIO_WAV)
print("RUN_DIR:", RUN_DIR)

Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

cmd = ["ffmpeg", "-hide_banner", "-y", "-i", AUDIO_IN]
if AUDIO_LIMIT_SECONDS:
    cmd += ["-t", str(AUDIO_LIMIT_SECONDS)]
cmd += ["-ac", "1", "-ar", "16000", AUDIO_WAV]
run_logged(cmd, "00_prepare_audio_ffmpeg.log", cwd="/kaggle/working", tail=15)


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

audio = AudioSegment.from_file(AUDIO_WAV)
print("Audio:", AUDIO_WAV)
print("Duration seconds:", len(audio) / 1000)
print("Frame rate:", audio.frame_rate)
print("Channels:", audio.channels)

preview_path = export_audio_preview(audio, Path(INPUT_DIR) / "preview_input_30s.wav", seconds=30)
print("Preview first 30s:", preview_path)
display(Audio(str(preview_path)))


## 6. Tải model phụ cho music clean và chuẩn bị MossFormer2 overlap separation

Bản này không clone SepReformer và không gọi HF audio-to-audio API ở stage 3. Stage 3 dùng package `clearvoice` và model local `MossFormer2_SS_16K`; model sẽ được ClearVoice tải về trong runtime Kaggle khi chạy lần đầu.


In [ ]:
from huggingface_hub import hf_hub_download

if RUN_DEMUCS:
    panns_path = hf_hub_download(
        repo_id="thelou1s/panns-inference",
        filename="Cnn14_mAP=0.431.pth",
        local_dir="/kaggle/working/sommelier/panns_data",
    )
    print("PANNs checkpoint:", panns_path)
else:
    print("RUN_DEMUCS=False, bỏ qua tải PANNs")

In [ ]:
import os
from pathlib import Path

print("Stage 03 mode: MossFormer2 self-host/local")
print("Model:", MOSSFORMER_MODEL_NAME)
print("Task:", MOSSFORMER_TASK)
print("Không clone/tải SepReformer local.")
print("Không gọi Hugging Face Inference API.")
print("Overlap mark threshold:", OVERLAP_MARK_THRESHOLD_SECONDS)
print("MossFormer min overlap:", MOSSFORMER_MIN_OVERLAP_SECONDS)
print("MossFormer context:", MOSSFORMER_CONTEXT_SECONDS)
print("Require different speaker:", OVERLAP_REQUIRE_DIFFERENT_SPEAKER)

Path(OVERLAP_DIR).mkdir(parents=True, exist_ok=True)
Path(f"{OVERLAP_DIR}/separated_segments").mkdir(parents=True, exist_ok=True)
Path(f"{OVERLAP_DIR}/mossformer_windows").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working/sommelier/podcast-pipeline")


## 7. Trace VAD chunking

Bước này chỉ để xem VAD chia audio thành các chunk dài thế nào trước diarization. Đây không phải output speaker segment cuối cùng.

Output:
- `/kaggle/working/run_full/01_diarization/trace_vad_chunks.json`
- `/kaggle/working/run_full/01_diarization/vad_chunks/*.wav`

In [ ]:

import os
import json
import shutil
from pathlib import Path
import pandas as pd
from pydub import AudioSegment
from IPython.display import display

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

import stage_common
import main_original_ASR_MoE as pipeline

cfg = pipeline.load_cfg("config.json")
logger = pipeline.Logger.get_logger()
pipeline.cfg = cfg
pipeline.logger = logger

device_name = "cuda" if pipeline.torch.cuda.is_available() else "cpu"
device = pipeline.torch.device(device_name)
pipeline.device_name = device_name
pipeline.device = device
pipeline.vad = pipeline.silero_vad.SileroVAD(device=device)

sample_rate = int(cfg["entrypoint"]["SAMPLE_RATE"])
audio_info = stage_common.load_audio_info(AUDIO_WAV, sample_rate)
diar_chunks, temp_chunk_dir = pipeline.prepare_diarization_chunks(AUDIO_WAV, audio_info)

chunk_dir = Path(DIAR_DIR) / "vad_chunks"
chunk_dir.mkdir(parents=True, exist_ok=True)

trace_chunks = []
for idx, chunk in enumerate(diar_chunks):
    src = Path(chunk["path"])
    dst = chunk_dir / f"chunk_{idx:03d}.wav"
    shutil.copy2(src, dst)
    duration = AudioSegment.from_file(dst).duration_seconds
    trace_chunks.append({
        "index": f"{idx:03d}",
        "path": str(dst),
        "offset": float(chunk["offset"]),
        "duration": float(duration),
        "start": float(chunk["offset"]),
        "end": float(chunk["offset"] + duration),
    })

if temp_chunk_dir:
    shutil.rmtree(temp_chunk_dir, ignore_errors=True)

stage_common.dump_json({
    "audio_path": AUDIO_WAV,
    "sample_rate": sample_rate,
    "chunks": trace_chunks,
    "metadata": {"stage": "vad_chunk_trace"},
}, Path(DIAR_DIR) / "trace_vad_chunks.json")

df_chunks = pd.DataFrame(trace_chunks)
print("VAD chunks:", len(df_chunks))
display(df_chunks.head(20))


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

if trace_chunks:
    first = trace_chunks[0]
    print(first)
    chunk_audio = AudioSegment.from_file(first["path"])
    preview_path = export_audio_preview(chunk_audio, Path(DIAR_DIR) / "preview_vad_chunk_0_30s.wav", seconds=30)
    print("Preview first 30s of chunk 0:", preview_path)
    display(Audio(str(preview_path)))


## 8. Stage 01 - Speaker diarization

Output: `/kaggle/working/run_full/01_diarization/diarization.json`.

Đây là bước Sortformer + speaker linking, tạo segment có `start`, `end`, `speaker`.

In [ ]:

import os
os.chdir("/kaggle/working/sommelier/podcast-pipeline")

run_logged([
    "python", "stage_01_diarize.py",
    "--input_audio", AUDIO_WAV,
    "--out", f"{DIAR_DIR}/diarization.json",
    "--merge_gap", str(SAME_SPEAKER_MERGE_GAP_SECONDS),
    "--max_segment_duration", "30.0",
    "--same_speaker_merge_gap", str(SAME_SPEAKER_MERGE_GAP_SECONDS),
    "--short_backchannel_seconds", str(SHORT_BACKCHANNEL_SECONDS),
    "--sortformer-pad-onset", "0.05",
    "--sortformer-pad-offset", "0.05",
], "18_stage_01_diarize.log", tail=25)


In [ ]:

import json
import pandas as pd
from IPython.display import display

with open(f"{DIAR_DIR}/diarization.json", "r", encoding="utf-8") as f:
    diar = json.load(f)

diar_segments = diar["segments"]
df_diar = pd.DataFrame(diar_segments)
df_diar["dur"] = df_diar["end"].astype(float) - df_diar["start"].astype(float)

print("File:", f"{DIAR_DIR}/diarization.json")
print("Total segments:", len(df_diar))
print("Speakers:", sorted(df_diar["speaker"].unique()) if len(df_diar) else [])
print("Duration median:", df_diar["dur"].median() if len(df_diar) else 0)
print("Duration mean:", df_diar["dur"].mean() if len(df_diar) else 0)
print("< 1s:", int((df_diar["dur"] < 1).sum()) if len(df_diar) else 0)
print("< 2s:", int((df_diar["dur"] < 2).sum()) if len(df_diar) else 0)
print("< 3s:", int((df_diar["dur"] < 3).sum()) if len(df_diar) else 0)

display(df_diar[["index", "start", "end", "dur", "speaker"]].head(30))


## 9. Stage 02 - Music/background clean

Output:
- `/kaggle/working/run_full/cleaned_audio.wav`
- `/kaggle/working/run_full/segment_flags.json`

In [ ]:

DEMUCS_ARG = "--demucs" if RUN_DEMUCS else "--no-demucs"

run_logged([
    "python", "stage_02_music_clean.py",
    "--input_audio", AUDIO_WAV,
    "--diarization_json", f"{DIAR_DIR}/diarization.json",
    "--out_audio", f"{MUSIC_DIR}/cleaned_audio.wav",
    "--out_flags", f"{MUSIC_DIR}/segment_flags.json",
    DEMUCS_ARG,
], "19_stage_02_music_clean.log", tail=25)


In [ ]:

from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

with open(f"{MUSIC_DIR}/segment_flags.json", "r", encoding="utf-8") as f:
    flags_data = json.load(f)

flags = flags_data.get("segment_demucs_flags", [])
print("Cleaned audio:", flags_data["audio_path"])
print("Segments:", len(flags_data["segments"]))
print("Demucs flagged segments:", sum(bool(x) for x in flags), "/", len(flags))

cleaned_preview = export_audio_preview(AudioSegment.from_file(f"{MUSIC_DIR}/cleaned_audio.wav"), Path(MUSIC_DIR) / "preview_cleaned_30s.wav", seconds=30)
print("Preview first 30s:", cleaned_preview)
display(Audio(str(cleaned_preview)))


## 10. Stage 03 - Overlap separation bằng MossFormer2 local

Output:
- `/kaggle/working/run_full/03_overlap/segments.json`
- `/kaggle/working/run_full/03_overlap/separated_segments/*.wav` cho segment đã thay vùng overlap bằng stem tách được.
- `/kaggle/working/run_full/03_overlap/mossformer_windows/*.wav` nếu bật `MOSSFORMER_SAVE_WINDOW_STEMS=True`, dùng để nghe kiểm tra stem gốc từ MossFormer.

Bước này:
1. Detect overlap từ timeline segment sau diarization/music-clean.
2. Mark overlap khác speaker từ `OVERLAP_MARK_THRESHOLD_SECONDS`; thử tách từ `MOSSFORMER_MIN_OVERLAP_SECONDS` để giữ micro backchannel trong podcast.
3. Cắt một cửa sổ quanh vùng overlap, thêm context trước/sau.
4. Gọi `ClearVoice(task='speech_separation', model_names=['MossFormer2_SS_16K'])` local/self-host.
5. Gán 2 stems về 2 speaker bằng speaker embedding; năng lượng chỉ còn là fallback/diagnostic.
6. Nếu assignment confidence thấp hoặc hai stem cùng giống một speaker, chỉ lưu stem để review và không ghi `enhanced_audio_path` cho ASR.

Các overlap micro, tách lỗi, hoặc assignment không chắc chắn vẫn được đánh dấu `needs_manual_review` để không đưa nhầm vào clean train.


In [ ]:
import contextlib
import os
import time
import traceback
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

import stage_common

try:
    from utils.stem_assignment import choose_stem_assignment
except Exception:
    def _fallback_score(source_scores, speaker):
        value = source_scores.get(speaker)
        return -1.0 if value is None else float(value)

    def _fallback_best_speaker(source_scores, speakers):
        available = [(speaker, _fallback_score(source_scores, speaker)) for speaker in speakers if speaker in source_scores]
        if not available:
            return None
        return max(available, key=lambda item: item[1])[0]

    def choose_stem_assignment(seg1_speaker, seg2_speaker, source_scores, *, min_confidence=0.55, min_margin=0.08):
        if len(source_scores) < 2:
            return {
                "accepted": False,
                "mapping": None,
                "assignment_method": "embedding_rejected",
                "reject_reasons": ["missing_source_scores"],
                "confidence": 0.0,
                "margin": 0.0,
                "direct_score": -1.0,
                "swapped_score": -1.0,
            }
        speakers = [str(seg1_speaker), str(seg2_speaker)]
        src0_scores, src1_scores = source_scores[0], source_scores[1]
        direct_components = [_fallback_score(src0_scores, speakers[0]), _fallback_score(src1_scores, speakers[1])]
        swapped_components = [_fallback_score(src0_scores, speakers[1]), _fallback_score(src1_scores, speakers[0])]
        direct_score = sum(direct_components)
        swapped_score = sum(swapped_components)
        if direct_score >= swapped_score:
            mapping = "direct"
            assignment_method = "embedding_direct"
            confidence = min(direct_components)
            margin = direct_score - swapped_score
        else:
            mapping = "swapped"
            assignment_method = "embedding_swapped"
            confidence = min(swapped_components)
            margin = swapped_score - direct_score
        src0_best = _fallback_best_speaker(src0_scores, speakers)
        src1_best = _fallback_best_speaker(src1_scores, speakers)
        reject_reasons = []
        if src0_best is None or src1_best is None:
            reject_reasons.append("missing_speaker_score")
        elif src0_best == src1_best:
            reject_reasons.append("same_best_speaker")
        if confidence < float(min_confidence):
            reject_reasons.append("confidence_lt_min")
        if margin < float(min_margin):
            reject_reasons.append("margin_lt_min")
        accepted = not reject_reasons
        return {
            "accepted": accepted,
            "mapping": mapping if accepted else None,
            "assignment_method": assignment_method if accepted else "embedding_rejected",
            "reject_reasons": reject_reasons,
            "confidence": round(float(max(confidence, 0.0)), 6),
            "margin": round(float(max(margin, 0.0)), 6),
            "direct_score": round(float(direct_score), 6),
            "swapped_score": round(float(swapped_score), 6),
            "source_best_speakers": [src0_best, src1_best],
        }


def _segment_index(segment, fallback_idx):
    return str(segment.get("index") or stage_common.normalized_index(fallback_idx))


def _detect_overlap_pairs(segments, threshold=0.1):
    indexed = [(idx, seg) for idx, seg in enumerate(segments)]
    indexed.sort(key=lambda item: float(item[1].get("start", 0.0)))
    pairs = []

    for left_pos, (idx1, seg1) in enumerate(indexed):
        start1 = float(seg1.get("start", 0.0))
        end1 = float(seg1.get("end", start1))
        for idx2, seg2 in indexed[left_pos + 1:]:
            start2 = float(seg2.get("start", 0.0))
            end2 = float(seg2.get("end", start2))
            if start2 >= end1:
                break

            overlap_start = max(start1, start2)
            overlap_end = min(end1, end2)
            overlap_duration = max(0.0, overlap_end - overlap_start)
            if overlap_duration < threshold:
                continue

            speaker1 = str(seg1.get("speaker", ""))
            speaker2 = str(seg2.get("speaker", ""))
            same_speaker = speaker1 == speaker2
            pairs.append({
                "seg1_idx": idx1,
                "seg2_idx": idx2,
                "seg1_index": _segment_index(seg1, idx1),
                "seg2_index": _segment_index(seg2, idx2),
                "seg1_speaker": speaker1,
                "seg2_speaker": speaker2,
                "same_speaker": same_speaker,
                "overlap_start": round(overlap_start, 6),
                "overlap_end": round(overlap_end, 6),
                "overlap_duration": round(overlap_duration, 6),
            })

    return pairs


def _detect_independent_osd_regions(audio_path):
    if not RUN_PYANNOTE_OSD:
        return [], ""
    try:
        from pyannote.audio import Pipeline

        hf_token = globals().get("token") or os.environ.get("HF_TOKEN")
        osd_pipeline = Pipeline.from_pretrained(PYANNOTE_OSD_MODEL, use_auth_token=hf_token)
        osd = osd_pipeline(str(audio_path))
        timeline = osd.get_timeline().support()
        regions = []
        for idx, turn in enumerate(timeline):
            duration = max(0.0, float(turn.end) - float(turn.start))
            if duration < PYANNOTE_OSD_MIN_DURATION_SECONDS:
                continue
            regions.append({
                "index": f"osd_{idx:05d}",
                "start": round(float(turn.start), 6),
                "end": round(float(turn.end), 6),
                "duration": round(duration, 6),
                "source": "pyannote_osd",
            })
        print("Independent pyannote OSD regions:", len(regions))
        return regions, ""
    except Exception as exc:
        print("Independent pyannote OSD failed:", repr(exc))
        if not PYANNOTE_OSD_FAIL_OPEN:
            raise
        return [], repr(exc)


def _pair_is_train_relevant(pair):
    if OVERLAP_REQUIRE_DIFFERENT_SPEAKER and pair.get("same_speaker"):
        return False
    return True


def _fit_length(wav, target_len):
    wav = np.asarray(wav, dtype=np.float32).reshape(-1)
    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    return wav[:target_len].astype(np.float32)


def _rms(wav):
    wav = np.asarray(wav, dtype=np.float32).reshape(-1)
    if wav.size == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(wav)) + 1e-12))


def _match_rms(source, reference):
    src_rms = _rms(source)
    ref_rms = _rms(reference)
    if src_rms <= 1e-6 or ref_rms <= 1e-6:
        return np.asarray(source, dtype=np.float32)
    gain = np.clip(ref_rms / src_rms, 0.25, 4.0)
    return np.clip(np.asarray(source, dtype=np.float32) * gain, -1.0, 1.0)


def _slice_by_time(wav, start, end, base_start, sample_rate):
    a = max(0, int(round((float(start) - float(base_start)) * sample_rate)))
    b = min(len(wav), int(round((float(end) - float(base_start)) * sample_rate)))
    if b <= a:
        return np.zeros(0, dtype=np.float32)
    return np.asarray(wav[a:b], dtype=np.float32)


def _energy_for_ranges(source, ranges, window_start, sample_rate):
    values = []
    for start, end in ranges:
        part = _slice_by_time(source, start, end, window_start, sample_rate)
        if part.size:
            values.append(_rms(part))
    return float(np.mean(values)) if values else 0.0


def _non_overlap_ranges(segment, pair, window_start, window_end):
    seg_start = max(float(segment.get("start", 0.0)), float(window_start))
    seg_end = min(float(segment.get("end", seg_start)), float(window_end))
    ov_start = float(pair["overlap_start"])
    ov_end = float(pair["overlap_end"])
    ranges = []
    if seg_start < min(ov_start, seg_end):
        ranges.append((seg_start, min(ov_start, seg_end)))
    if max(ov_end, seg_start) < seg_end:
        ranges.append((max(ov_end, seg_start), seg_end))
    return ranges


def _extract_sources(clearvoice_model, window_audio, sample_rate):
    if sample_rate != 16000:
        window_audio = librosa.resample(window_audio, orig_sr=sample_rate, target_sr=16000).astype(np.float32)
        sample_rate = 16000
    audio = np.asarray(window_audio, dtype=np.float32).reshape(1, -1)
    output = clearvoice_model(audio, False)
    output = np.asarray(output, dtype=np.float32)

    if output.ndim == 3 and output.shape[0] >= 2:
        # ClearVoice speech separation demo documents output as [spk, batch, length].
        sources = [output[i, 0, :] for i in range(min(2, output.shape[0]))]
    elif output.ndim == 3 and output.shape[1] >= 2:
        # Defensive fallback if a future version returns [batch, spk, length].
        sources = [output[0, i, :] for i in range(min(2, output.shape[1]))]
    elif output.ndim == 2 and output.shape[0] >= 2:
        sources = [output[0, :], output[1, :]]
    else:
        raise RuntimeError(f"Unexpected MossFormer output shape: {output.shape}")

    if len(sources) < 2:
        raise RuntimeError(f"MossFormer returned fewer than 2 sources: {output.shape}")
    return _fit_length(sources[0], len(window_audio)), _fit_length(sources[1], len(window_audio))


def _embedding_device():
    import torch

    requested = str(globals().get("MOSSFORMER_EMBEDDING_DEVICE", "cpu")).strip().lower()
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested.startswith("cuda") and not torch.cuda.is_available():
        requested = "cpu"
    return torch.device(requested)


def _load_speaker_embedding_model():
    if not MOSSFORMER_USE_SPEAKER_EMBEDDING_ASSIGNMENT:
        return None, None, "disabled"
    try:
        from pyannote.audio import Model as PyannoteModel

        hf_token = globals().get("token") or os.environ.get("HF_TOKEN")
        device = _embedding_device()
        model = PyannoteModel.from_pretrained("pyannote/embedding", use_auth_token=hf_token)
        model = model.to(device)
        model.eval()
        print("Loaded speaker embedding model on", device)
        return model, device, ""
    except Exception as exc:
        print("Speaker embedding model failed:", repr(exc))
        if not MOSSFORMER_EMBEDDING_FAIL_OPEN:
            raise
        return None, None, repr(exc)


def _embedding_from_audio(audio, sample_rate, embedding_model, embedding_device):
    import torch

    audio = np.asarray(audio, dtype=np.float32).reshape(-1)
    if sample_rate != 16000:
        audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=16000).astype(np.float32)
    min_len = int(16000 * 0.75)
    if audio.size < min_len:
        audio = np.pad(audio, (0, min_len - audio.size))
    tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).to(embedding_device)
    with torch.inference_mode():
        embedding = embedding_model(tensor)
    return _embedding_vector(embedding)


def _embedding_vector(embedding):
    embedding = embedding.detach().float()
    while embedding.ndim > 1:
        embedding = embedding.mean(dim=0)
    return embedding


def _cosine_score(left, right):
    import torch

    return float(torch.nn.functional.cosine_similarity(left, right, dim=0).item())


def _build_reference_embeddings(segments, speaker_pairs, waveform, sample_rate, embedding_model, embedding_device):
    if embedding_model is None:
        return {}, {}

    import torch

    overlapped_indices = set()
    for pair in speaker_pairs:
        overlapped_indices.add(pair["seg1_idx"])
        overlapped_indices.add(pair["seg2_idx"])

    speakers = sorted({str(seg.get("speaker", "")) for seg in segments if str(seg.get("speaker", ""))})
    reference_embeddings = {}
    reference_counts = {}
    for speaker in speakers:
        candidates = []
        for idx, seg in enumerate(segments):
            if idx in overlapped_indices:
                continue
            if str(seg.get("speaker", "")) != speaker:
                continue
            if bool(seg.get("is_short_backchannel")):
                continue
            dur = float(seg.get("end", 0.0)) - float(seg.get("start", 0.0))
            if dur < MOSSFORMER_REFERENCE_MIN_SECONDS:
                continue
            candidates.append((dur, idx, seg))
        candidates.sort(reverse=True)

        vectors = []
        for _, _, seg in candidates[:MOSSFORMER_REFERENCE_MAX_SEGMENTS_PER_SPEAKER]:
            a = max(0, int(round(float(seg.get("start", 0.0)) * sample_rate)))
            b = min(len(waveform), int(round(float(seg.get("end", 0.0)) * sample_rate)))
            if b <= a:
                continue
            try:
                vectors.append(_embedding_from_audio(waveform[a:b], sample_rate, embedding_model, embedding_device))
            except Exception as exc:
                print(f"Reference embedding failed for {speaker}:", repr(exc))
        if vectors:
            reference_embeddings[speaker] = torch.stack(vectors).mean(dim=0)
            reference_counts[speaker] = len(vectors)

    print("Speaker reference embeddings:", reference_counts)
    return reference_embeddings, reference_counts


def _energy_assignment(src0, src1, seg1, seg2, pair, window_start, window_end, sample_rate):
    seg1_ranges = _non_overlap_ranges(seg1, pair, window_start, window_end)
    seg2_ranges = _non_overlap_ranges(seg2, pair, window_start, window_end)

    src0_seg1 = _energy_for_ranges(src0, seg1_ranges, window_start, sample_rate)
    src1_seg1 = _energy_for_ranges(src1, seg1_ranges, window_start, sample_rate)
    src0_seg2 = _energy_for_ranges(src0, seg2_ranges, window_start, sample_rate)
    src1_seg2 = _energy_for_ranges(src1, seg2_ranges, window_start, sample_rate)

    score_direct = src0_seg1 + src1_seg2
    score_swapped = src1_seg1 + src0_seg2
    scores = {
        "energy_src0_seg1": round(float(src0_seg1), 6),
        "energy_src1_seg1": round(float(src1_seg1), 6),
        "energy_src0_seg2": round(float(src0_seg2), 6),
        "energy_src1_seg2": round(float(src1_seg2), 6),
        "energy_direct_score": round(float(score_direct), 6),
        "energy_swapped_score": round(float(score_swapped), 6),
    }
    if score_swapped > score_direct:
        return src1, src0, "energy_swapped", scores
    return src0, src1, "energy_direct", scores


def _assign_sources_to_pair(
    src0,
    src1,
    seg1,
    seg2,
    pair,
    window_start,
    window_end,
    sample_rate,
    reference_embeddings=None,
    embedding_model=None,
    embedding_device=None,
):
    energy_seg1, energy_seg2, energy_method, energy_scores = _energy_assignment(
        src0, src1, seg1, seg2, pair, window_start, window_end, sample_rate
    )

    seg1_speaker = str(seg1.get("speaker", ""))
    seg2_speaker = str(seg2.get("speaker", ""))
    decision = {
        "accepted": False,
        "mapping": None,
        "assignment_method": energy_method if not MOSSFORMER_REQUIRE_EMBEDDING_ASSIGNMENT else "embedding_unavailable",
        "reject_reasons": [],
        "confidence": 0.0,
        "margin": 0.0,
        **energy_scores,
    }

    refs = reference_embeddings or {}
    can_use_embedding = (
        MOSSFORMER_USE_SPEAKER_EMBEDDING_ASSIGNMENT
        and embedding_model is not None
        and embedding_device is not None
        and seg1_speaker in refs
        and seg2_speaker in refs
    )

    if can_use_embedding:
        src0_embedding = _embedding_from_audio(src0, sample_rate, embedding_model, embedding_device)
        src1_embedding = _embedding_from_audio(src1, sample_rate, embedding_model, embedding_device)
        source_scores = [
            {
                seg1_speaker: _cosine_score(src0_embedding, refs[seg1_speaker]),
                seg2_speaker: _cosine_score(src0_embedding, refs[seg2_speaker]),
            },
            {
                seg1_speaker: _cosine_score(src1_embedding, refs[seg1_speaker]),
                seg2_speaker: _cosine_score(src1_embedding, refs[seg2_speaker]),
            },
        ]
        decision = choose_stem_assignment(
            seg1_speaker,
            seg2_speaker,
            source_scores,
            min_confidence=MOSSFORMER_ASSIGNMENT_MIN_CONFIDENCE,
            min_margin=MOSSFORMER_ASSIGNMENT_MIN_MARGIN,
        )
        decision.update(energy_scores)
        decision["source_scores"] = source_scores
        if decision["accepted"]:
            if decision["mapping"] == "swapped":
                return src1, src0, decision
            return src0, src1, decision
        return energy_seg1, energy_seg2, decision

    reason = "reference_embedding_missing" if embedding_model is not None else "embedding_model_unavailable"
    decision["reject_reasons"] = [reason]
    decision["accepted"] = not MOSSFORMER_REQUIRE_EMBEDDING_ASSIGNMENT
    if decision["accepted"]:
        decision["mapping"] = "swapped" if energy_method == "energy_swapped" else "direct"
    return energy_seg1, energy_seg2, decision


def _write_wav(path, wav, sample_rate):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), np.asarray(wav, dtype=np.float32), int(sample_rate))
    return str(path.resolve())


def _write_stage3_mossformer():
    start_time = time.time()
    segment_data = stage_common.load_json(f"{MUSIC_DIR}/segment_flags.json")
    segments = segment_data.get("segments", [])
    sample_rate = int(segment_data.get("sample_rate") or 16000)
    cleaned_audio_path = Path(MUSIC_DIR) / "cleaned_audio.wav"
    waveform, sr = librosa.load(str(cleaned_audio_path), sr=sample_rate, mono=True)
    waveform = np.asarray(waveform, dtype=np.float32)
    audio_duration = len(waveform) / float(sample_rate) if sample_rate else 0.0

    temporal_pairs = _detect_overlap_pairs(segments, threshold=OVERLAP_MARK_THRESHOLD_SECONDS)
    speaker_pairs = [pair for pair in temporal_pairs if _pair_is_train_relevant(pair)]
    review_pairs = [pair for pair in speaker_pairs if float(pair["overlap_duration"]) >= OVERLAP_REVIEW_THRESHOLD_SECONDS]
    independent_osd_regions, osd_error = _detect_independent_osd_regions(cleaned_audio_path)

    regions_by_segment = {idx: [] for idx in range(len(segments))}
    osd_regions_by_segment = {idx: [] for idx in range(len(segments))}
    for pair in speaker_pairs:
        region1 = {
            "with_index": pair["seg2_index"],
            "with_speaker": pair["seg2_speaker"],
            "start": pair["overlap_start"],
            "end": pair["overlap_end"],
            "duration": pair["overlap_duration"],
            "review_required": float(pair["overlap_duration"]) >= OVERLAP_REVIEW_THRESHOLD_SECONDS,
        }
        region2 = {
            "with_index": pair["seg1_index"],
            "with_speaker": pair["seg1_speaker"],
            "start": pair["overlap_start"],
            "end": pair["overlap_end"],
            "duration": pair["overlap_duration"],
            "review_required": float(pair["overlap_duration"]) >= OVERLAP_REVIEW_THRESHOLD_SECONDS,
        }
        regions_by_segment[pair["seg1_idx"]].append(region1)
        regions_by_segment[pair["seg2_idx"]].append(region2)

    for osd_region in independent_osd_regions:
        osd_start = float(osd_region["start"])
        osd_end = float(osd_region["end"])
        for idx, seg in enumerate(segments):
            seg_start = float(seg.get("start", 0.0))
            seg_end = float(seg.get("end", seg_start))
            ov_start = max(seg_start, osd_start)
            ov_end = min(seg_end, osd_end)
            ov_duration = max(0.0, ov_end - ov_start)
            if ov_duration < PYANNOTE_OSD_MIN_DURATION_SECONDS:
                continue
            payload = dict(osd_region)
            payload.update({
                "segment_overlap_start": round(ov_start, 6),
                "segment_overlap_end": round(ov_end, 6),
                "segment_overlap_duration": round(ov_duration, 6),
            })
            osd_regions_by_segment[idx].append(payload)

    enhanced_by_idx = {}
    replacement_counts = {idx: 0 for idx in range(len(segments))}
    separated_pairs = 0
    low_confidence_pairs = 0
    failed_pairs = 0
    skipped_pairs = 0
    clearvoice_model = None
    embedding_model = None
    embedding_device = None
    embedding_error = ""
    reference_embeddings = {}
    reference_embedding_counts = {}
    error = None

    def get_segment_audio(idx):
        if idx not in enhanced_by_idx:
            seg = segments[idx]
            a = max(0, int(round(float(seg.get("start", 0.0)) * sample_rate)))
            b = min(len(waveform), int(round(float(seg.get("end", 0.0)) * sample_rate)))
            enhanced_by_idx[idx] = waveform[a:b].copy()
        return enhanced_by_idx[idx]

    try:
        if RUN_MOSSFORMER_SEPARATION:
            from clearvoice import ClearVoice
            clearvoice_model = ClearVoice(task=MOSSFORMER_TASK, model_names=[MOSSFORMER_MODEL_NAME])
            print("Loaded ClearVoice model:", MOSSFORMER_MODEL_NAME)
        else:
            print("RUN_MOSSFORMER_SEPARATION=False, chỉ mark overlap.")
    except Exception as exc:
        error = exc
        print("MossFormer load failed:", repr(exc))
        if not MOSSFORMER_FAIL_OPEN:
            raise

    if clearvoice_model is not None:
        embedding_model, embedding_device, embedding_error = _load_speaker_embedding_model()
        reference_embeddings, reference_embedding_counts = _build_reference_embeddings(
            segments, speaker_pairs, waveform, sample_rate, embedding_model, embedding_device
        )

    separated_dir = Path(OVERLAP_DIR) / "separated_segments"
    windows_dir = Path(OVERLAP_DIR) / "mossformer_windows"
    separated_dir.mkdir(parents=True, exist_ok=True)
    windows_dir.mkdir(parents=True, exist_ok=True)

    for pair_idx, pair in enumerate(speaker_pairs):
        pair["separation_status"] = "marked_only"
        pair["skip_reasons"] = []
        if clearvoice_model is None:
            pair["skip_reasons"].append("model_unavailable")
            skipped_pairs += 1
            continue

        seg1 = segments[pair["seg1_idx"]]
        seg2 = segments[pair["seg2_idx"]]
        seg1_dur = float(seg1.get("end", 0.0)) - float(seg1.get("start", 0.0))
        seg2_dur = float(seg2.get("end", 0.0)) - float(seg2.get("start", 0.0))
        overlap_dur = float(pair["overlap_duration"])

        if overlap_dur < MOSSFORMER_MIN_OVERLAP_SECONDS:
            pair["skip_reasons"].append("overlap_lt_min")
        if min(seg1_dur, seg2_dur) < MOSSFORMER_MIN_SEGMENT_SECONDS:
            pair["skip_reasons"].append("segment_lt_min")
        if pair["skip_reasons"]:
            pair["separation_status"] = "skipped"
            skipped_pairs += 1
            continue

        overlap_start = float(pair["overlap_start"])
        overlap_end = float(pair["overlap_end"])
        union_start = min(float(seg1.get("start", 0.0)), float(seg2.get("start", 0.0)))
        union_end = max(float(seg1.get("end", 0.0)), float(seg2.get("end", 0.0)))
        window_start = max(union_start, overlap_start - MOSSFORMER_CONTEXT_SECONDS, 0.0)
        window_end = min(union_end, overlap_end + MOSSFORMER_CONTEXT_SECONDS, audio_duration)
        if window_end - window_start > MOSSFORMER_MAX_WINDOW_SECONDS:
            center = (overlap_start + overlap_end) / 2.0
            half = MOSSFORMER_MAX_WINDOW_SECONDS / 2.0
            window_start = max(0.0, center - half)
            window_end = min(audio_duration, center + half)

        a = max(0, int(round(window_start * sample_rate)))
        b = min(len(waveform), int(round(window_end * sample_rate)))
        window_audio = waveform[a:b]
        if len(window_audio) < int(0.1 * sample_rate):
            pair["separation_status"] = "skipped"
            pair["skip_reasons"].append("window_too_short")
            skipped_pairs += 1
            continue

        try:
            print(f"MossFormer pair {pair_idx}: {pair['seg1_index']} x {pair['seg2_index']} overlap={overlap_dur:.3f}s window={window_start:.3f}-{window_end:.3f}")
            src0, src1 = _extract_sources(clearvoice_model, window_audio, sample_rate)
            seg1_src, seg2_src, assignment_decision = _assign_sources_to_pair(
                src0,
                src1,
                seg1,
                seg2,
                pair,
                window_start,
                window_end,
                sample_rate,
                reference_embeddings=reference_embeddings,
                embedding_model=embedding_model,
                embedding_device=embedding_device,
            )

            pair["assignment_method"] = assignment_decision.get("assignment_method", "unknown")
            pair["assignment_accepted"] = bool(assignment_decision.get("accepted"))
            pair["assignment_confidence"] = assignment_decision.get("confidence", 0.0)
            pair["assignment_margin"] = assignment_decision.get("margin", 0.0)
            pair["assignment_reject_reasons"] = assignment_decision.get("reject_reasons", [])
            pair["assignment_decision"] = assignment_decision
            pair["window_start"] = round(window_start, 6)
            pair["window_end"] = round(window_end, 6)

            if MOSSFORMER_SAVE_WINDOW_STEMS:
                pair_prefix = f"pair{pair_idx:03d}_{pair['seg1_index']}_{pair['seg2_index']}"
                pair["window_audio_path"] = _write_wav(windows_dir / f"{pair_prefix}_mix.wav", window_audio, sample_rate)
                pair["raw_stem1_audio_path"] = _write_wav(windows_dir / f"{pair_prefix}_raw_stem1.wav", src0, sample_rate)
                pair["raw_stem2_audio_path"] = _write_wav(windows_dir / f"{pair_prefix}_raw_stem2.wav", src1, sample_rate)
                pair["stem1_audio_path"] = pair["raw_stem1_audio_path"]
                pair["stem2_audio_path"] = pair["raw_stem2_audio_path"]
                if assignment_decision.get("accepted"):
                    pair["assigned_seg1_audio_path"] = _write_wav(windows_dir / f"{pair_prefix}_assigned_seg1.wav", seg1_src, sample_rate)
                    pair["assigned_seg2_audio_path"] = _write_wav(windows_dir / f"{pair_prefix}_assigned_seg2.wav", seg2_src, sample_rate)

            if not assignment_decision.get("accepted"):
                pair["separation_status"] = "low_confidence"
                pair["skip_reasons"].append("assignment_low_confidence")
                low_confidence_pairs += 1
                if MOSSFORMER_DISABLE_LOW_CONFIDENCE_ASR:
                    continue

            ov_a = int(round((overlap_start - window_start) * sample_rate))
            ov_b = int(round((overlap_end - window_start) * sample_rate))
            ref_overlap = window_audio[ov_a:ov_b]
            seg1_overlap = _match_rms(seg1_src[ov_a:ov_b], ref_overlap)
            seg2_overlap = _match_rms(seg2_src[ov_a:ov_b], ref_overlap)

            for seg_idx, separated_overlap in [(pair["seg1_idx"], seg1_overlap), (pair["seg2_idx"], seg2_overlap)]:
                seg_audio = get_segment_audio(seg_idx)
                seg_start = float(segments[seg_idx].get("start", 0.0))
                local_a = max(0, int(round((overlap_start - seg_start) * sample_rate)))
                local_b = min(len(seg_audio), int(round((overlap_end - seg_start) * sample_rate)))
                if local_b > local_a:
                    seg_audio[local_a:local_b] = _fit_length(separated_overlap, local_b - local_a)
                    replacement_counts[seg_idx] += 1

            pair["separation_status"] = "separated"
            separated_pairs += 1
        except Exception as exc:
            pair["separation_status"] = "failed"
            pair["error"] = repr(exc)
            failed_pairs += 1
            print("MossFormer pair failed:", repr(exc))
            if not MOSSFORMER_FAIL_OPEN:
                raise

    pair_statuses_by_segment = {idx: [] for idx in range(len(segments))}
    for pair in speaker_pairs:
        base_status = {
            "separation_status": pair.get("separation_status", "marked_only"),
            "assignment_method": pair.get("assignment_method", ""),
            "assignment_accepted": bool(pair.get("assignment_accepted", False)),
            "assignment_confidence": pair.get("assignment_confidence", 0.0),
            "assignment_margin": pair.get("assignment_margin", 0.0),
            "assignment_reject_reasons": pair.get("assignment_reject_reasons", []),
        }
        pair_statuses_by_segment[pair["seg1_idx"]].append({**base_status, "with_index": pair["seg2_index"], "with_speaker": pair["seg2_speaker"]})
        pair_statuses_by_segment[pair["seg2_idx"]].append({**base_status, "with_index": pair["seg1_index"], "with_speaker": pair["seg1_speaker"]})

    clean_segments = []
    separated_segments_count = 0
    for idx, segment in enumerate(segments):
        clean_segment = stage_common.clean_segment_for_json(segment)
        regions = regions_by_segment.get(idx, [])
        independent_regions = osd_regions_by_segment.get(idx, [])
        overlap_total = round(sum(float(region["duration"]) for region in regions), 6)
        needs_review = any(bool(region.get("review_required")) for region in regions) or bool(independent_regions)
        is_short_backchannel = bool(clean_segment.get("is_short_backchannel"))
        base_label = "short_backchannel_review" if is_short_backchannel else "clean_candidate"
        if independent_regions and not regions:
            base_label = "independent_osd_review"

        clean_segment["has_overlap"] = bool(regions) or bool(independent_regions)
        clean_segment["overlap_count"] = len(regions)
        clean_segment["overlap_total_seconds"] = overlap_total
        clean_segment["overlap_regions"] = regions
        clean_segment["independent_osd_regions"] = independent_regions
        clean_segment["has_independent_osd_overlap"] = bool(independent_regions)
        clean_segment["overlap_pair_statuses"] = pair_statuses_by_segment.get(idx, [])
        clean_segment["low_confidence_overlap_count"] = sum(
            1 for item in pair_statuses_by_segment.get(idx, []) if item.get("separation_status") == "low_confidence"
        )
        clean_segment["needs_manual_review"] = bool(regions) or is_short_backchannel or bool(independent_regions)
        clean_segment["exclude_from_clean_train"] = bool(regions) or is_short_backchannel or bool(independent_regions)
        clean_segment["train_quality_label"] = "overlap_review" if regions else base_label
        if regions and not needs_review:
            clean_segment["train_quality_label"] = "short_overlap_review"

        if replacement_counts.get(idx, 0) > 0 and idx in enhanced_by_idx:
            seg_index = clean_segment.get("index") or stage_common.normalized_index(idx)
            speaker = clean_segment.get("speaker", "Unknown")
            out_path = separated_dir / f"{seg_index}_{speaker}_mossformer.wav"
            clean_segment["enhanced_audio_path"] = _write_wav(out_path, enhanced_by_idx[idx], sample_rate)
            clean_segment["is_separated"] = True
            clean_segment["separation_method"] = "mossformer2_ss_16k"
            clean_segment["separation_status"] = "separated"
            clean_segment["separation_replacements"] = int(replacement_counts[idx])
            clean_segment["train_quality_label"] = "micro_separated_review" if overlap_total < 0.5 else "separated_overlap_review"
            separated_segments_count += 1
        else:
            clean_segment["enhanced_audio_path"] = ""
            clean_segment["is_separated"] = False
            clean_segment["separation_method"] = "mossformer2_ss_16k" if regions else "none"
            if clean_segment.get("low_confidence_overlap_count", 0) > 0:
                clean_segment["separation_status"] = "low_confidence"
            else:
                clean_segment["separation_status"] = "marked_only" if regions else "none"
            clean_segment["separation_replacements"] = 0

        clean_segments.append(clean_segment)

    configured_main_speakers = MAIN_SPEAKERS if MAIN_SPEAKERS else None
    clean_segments, duplex_grouping = stage_common.assign_duplex_train_groups(
        clean_segments,
        expected_main_speakers=EXPECTED_MAIN_SPEAKERS,
        main_speakers=configured_main_speakers,
    )

    processing_time = time.time() - start_time
    out_data = {
        "audio_path": str(cleaned_audio_path),
        "source_audio_path": segment_data.get("source_audio_path") or str(cleaned_audio_path),
        "audio_name": segment_data.get("audio_name") or stage_common.audio_name_from_path(cleaned_audio_path),
        "sample_rate": sample_rate,
        "audio_duration_seconds": audio_duration,
        "segments": clean_segments,
        "segment_demucs_flags": segment_data.get("segment_demucs_flags", [False] * len(clean_segments)),
        "overlap_pairs": speaker_pairs,
        "temporal_overlap_pairs": temporal_pairs,
        "metadata": {
            "stage": "overlap_separate_mossformer2",
            "enabled": True,
            "separation_enabled": bool(RUN_MOSSFORMER_SEPARATION),
            "separator": "clearvoice_mossformer2_ss_16k",
            "model_name": MOSSFORMER_MODEL_NAME,
            "task": MOSSFORMER_TASK,
            "processing_time_seconds": processing_time,
            "overlap_mark_threshold_seconds": OVERLAP_MARK_THRESHOLD_SECONDS,
            "overlap_review_threshold_seconds": OVERLAP_REVIEW_THRESHOLD_SECONDS,
            "overlap_require_different_speaker": OVERLAP_REQUIRE_DIFFERENT_SPEAKER,
            "mossformer_min_overlap_seconds": MOSSFORMER_MIN_OVERLAP_SECONDS,
            "mossformer_min_segment_seconds": MOSSFORMER_MIN_SEGMENT_SECONDS,
            "mossformer_context_seconds": MOSSFORMER_CONTEXT_SECONDS,
            "mossformer_max_window_seconds": MOSSFORMER_MAX_WINDOW_SECONDS,
            "mossformer_use_speaker_embedding_assignment": bool(MOSSFORMER_USE_SPEAKER_EMBEDDING_ASSIGNMENT),
            "mossformer_embedding_device": MOSSFORMER_EMBEDDING_DEVICE,
            "mossformer_embedding_error": embedding_error,
            "mossformer_reference_min_seconds": MOSSFORMER_REFERENCE_MIN_SECONDS,
            "mossformer_reference_counts": reference_embedding_counts,
            "mossformer_assignment_min_confidence": MOSSFORMER_ASSIGNMENT_MIN_CONFIDENCE,
            "mossformer_assignment_min_margin": MOSSFORMER_ASSIGNMENT_MIN_MARGIN,
            "mossformer_require_embedding_assignment": bool(MOSSFORMER_REQUIRE_EMBEDDING_ASSIGNMENT),
            "mossformer_disable_low_confidence_asr": bool(MOSSFORMER_DISABLE_LOW_CONFIDENCE_ASR),
            "expected_main_speakers": EXPECTED_MAIN_SPEAKERS,
            "configured_main_speakers": configured_main_speakers or [],
            "duplex_train_grouping": duplex_grouping,
            "temporal_overlap_pairs_count": len(temporal_pairs),
            "speaker_overlap_pairs_count": len(speaker_pairs),
            "review_overlap_pairs_count": len(review_pairs),
            "run_pyannote_osd": bool(RUN_PYANNOTE_OSD),
            "pyannote_osd_model": PYANNOTE_OSD_MODEL,
            "pyannote_osd_regions_count": len(independent_osd_regions),
            "pyannote_osd_error": osd_error,
            "separated_pairs_count": separated_pairs,
            "low_confidence_pairs_count": low_confidence_pairs,
            "failed_pairs_count": failed_pairs,
            "skipped_pairs_count": skipped_pairs,
            "separated_segments_count": separated_segments_count,
            "error": str(error) if error else "",
        },
    }
    stage_common.dump_json(out_data, Path(OVERLAP_DIR) / "segments.json")

    print("Stage 03 MossFormer complete:", Path(OVERLAP_DIR) / "segments.json")
    print("Segments:", len(clean_segments))
    print("Temporal overlap pairs:", len(temporal_pairs))
    print("Cross-speaker overlap pairs:", len(speaker_pairs))
    print("Review overlap pairs:", len(review_pairs))
    print("Independent OSD regions:", len(independent_osd_regions))
    print("Separated pairs:", separated_pairs)
    print("Low-confidence pairs:", low_confidence_pairs)
    print("Separated segments:", separated_segments_count)
    print("Failed pairs:", failed_pairs)
    print("Skipped pairs:", skipped_pairs)
    print("Main speakers:", duplex_grouping.get("main_speakers", []))
    print("Duplex group counts:", duplex_grouping.get("group_counts", {}))


log_path = Path(LOG_DIR_PATH) / "20_stage_03_overlap_separate.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
with open(log_path, "w", encoding="utf-8") as log_file:
    with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
        try:
            _write_stage3_mossformer()
        except Exception:
            traceback.print_exc()
            raise

print("Log:", log_path)
print(tail_file(log_path, 100))


In [ ]:
with open(f"{OVERLAP_DIR}/segments.json", "r", encoding="utf-8") as f:
    seg_data = json.load(f)

segments = seg_data["segments"]
metadata = seg_data.get("metadata", {})
df_seg = pd.DataFrame(segments)
df_seg["dur"] = df_seg["end"].astype(float) - df_seg["start"].astype(float)

print("Segments:", len(df_seg))
print("Stage 03 mode:", metadata.get("stage"))
print("Temporal overlap pairs:", metadata.get("temporal_overlap_pairs_count", 0))
print("Cross-speaker overlap pairs:", metadata.get("speaker_overlap_pairs_count", 0))
print("Separated pairs:", metadata.get("separated_pairs_count", 0))
print("Failed pairs:", metadata.get("failed_pairs_count", 0))
print("Skipped pairs:", metadata.get("skipped_pairs_count", 0))
print("Segments needing review:", int(df_seg.get("needs_manual_review", pd.Series(dtype=bool)).fillna(False).sum()) if len(df_seg) else 0)
print("Separated segments:", int(df_seg.get("is_separated", pd.Series(dtype=bool)).fillna(False).sum()) if len(df_seg) else 0)
print("Main speakers:", metadata.get("duplex_train_grouping", {}).get("main_speakers", []))
print("Duplex group counts:", metadata.get("duplex_train_grouping", {}).get("group_counts", {}))
print("Duration median:", df_seg["dur"].median() if len(df_seg) else 0)
display(df_seg[[c for c in ["index", "start", "end", "dur", "speaker", "has_overlap", "overlap_count", "overlap_total_seconds", "needs_manual_review", "train_quality_label", "is_separated", "separation_method", "separation_status", "separation_replacements", "low_confidence_overlap_count", "duplex_train_group", "duplex_group_reason", "is_main_speaker", "enhanced_audio_path"] if c in df_seg.columns]].head(30))


In [ ]:
from pathlib import Path
import json
import pandas as pd
from pydub import AudioSegment
from IPython.display import Audio, display, HTML

segments_path = Path(OVERLAP_DIR) / "segments.json"
cleaned_audio_path = Path(MUSIC_DIR) / "cleaned_audio.wav"

with open(segments_path, "r", encoding="utf-8") as f:
    stage3 = json.load(f)

segments = stage3["segments"]
cleaned_audio = AudioSegment.from_file(cleaned_audio_path)

df = pd.DataFrame([
    {
        "i": i,
        "index": s.get("index"),
        "start": s.get("start"),
        "end": s.get("end"),
        "speaker": s.get("speaker"),
        "has_overlap": s.get("has_overlap", False),
        "overlap_count": s.get("overlap_count", 0),
        "overlap_total_seconds": s.get("overlap_total_seconds", 0),
        "needs_manual_review": s.get("needs_manual_review", False),
        "train_quality_label": s.get("train_quality_label", ""),
        "is_separated": s.get("is_separated", False),
        "separation_status": s.get("separation_status", ""),
        "duplex_train_group": s.get("duplex_train_group", ""),
        "duplex_group_reason": s.get("duplex_group_reason", ""),
        "enhanced_audio_path": s.get("enhanced_audio_path", ""),
    }
    for i, s in enumerate(segments)
])

print("Stage 03 segments:", len(df))
print("Overlap-marked segments:", int(df["has_overlap"].sum()) if len(df) else 0)
print("Separated segments:", int(df["is_separated"].sum()) if len(df) else 0)
display(df.head(20))


def listen_stage3_segment(i):
    s = segments[i]
    start = float(s["start"])
    end = float(s["end"])
    speaker = s.get("speaker", "UNKNOWN")
    regions = s.get("overlap_regions", []) or []
    enhanced_path = s.get("enhanced_audio_path", "")

    print(f"Segment {i} / {s.get('index', '')}")
    print(f"Speaker: {speaker}")
    print(f"Time: {start:.2f}s - {end:.2f}s")
    print(f"Has overlap: {bool(regions)}")
    print(f"Separated: {s.get('is_separated', False)}")
    print(f"Separation status: {s.get('separation_status', '')}")
    print(f"Train quality label: {s.get('train_quality_label', '')}")
    if regions:
        display(pd.DataFrame(regions))

    tmp_clip = Path(PREVIEW_DIR) / f"preview_stage3_cleaned_{i:04d}.wav"
    cleaned_audio[int(start * 1000):int(end * 1000)].export(tmp_clip, format="wav")

    display(HTML("<b>Cleaned audio slice sau Stage 02:</b>"))
    display(Audio(str(tmp_clip)))

    if enhanced_path and Path(enhanced_path).exists():
        display(HTML("<b>MossFormer enhanced segment dùng cho ASR:</b>"))
        display(Audio(enhanced_path))
    else:
        print("Không có enhanced_audio_path cho segment này.")

print("Call listen_stage3_segment(i) để nghe thủ công, ví dụ: listen_stage3_segment(0)")


## 11. Cài cuDNN 8 riêng cho faster-whisper/ctranslate2

Không cài đè vào global torch. Chỉ cài vào `/kaggle/working/cudnn8` rồi truyền `LD_LIBRARY_PATH` khi chạy ASR.


In [ ]:
import shutil
from pathlib import Path

cudnn_dir = Path("/kaggle/working/cudnn8")
if cudnn_dir.exists():
    shutil.rmtree(cudnn_dir)

run_logged([
    "python", "-m", "pip", "install", "--target", str(cudnn_dir),
    "nvidia-cudnn-cu12==8.9.7.29",
], "21_pip_cudnn8.log", cwd="/kaggle/working/sommelier/podcast-pipeline", tail=20)

matches = sorted(cudnn_dir.rglob("libcudnn_ops_infer.so.8"))
print("libcudnn matches:", len(matches))
for p in matches[:5]:
    print(p)


## 11.5 PhoWhisper local/self-host mode

Bản này không patch PhoWhisper sang Hugging Face API. PhoWhisper sẽ tải model về runtime Kaggle và chạy local trong stage 4.


In [ ]:
from pathlib import Path
import re

asr_module_path = Path("/kaggle/working/sommelier/podcast-pipeline/stage_04_asr.py")
if not asr_module_path.exists():
    raise FileNotFoundError(asr_module_path)

print("PhoWhisper local/self-host mode")
print("PHOWHISPER_USE_HF_API:", PHOWHISPER_USE_HF_API)
print("Stage 04 sẽ set env PHOWHISPER_USE_HF_API=0 để không gọi HF Inference API.")

if USE_WHISPER_INITIAL_PROMPT:
    src = asr_module_path.read_text(encoding="utf-8")
    prompt_literal = repr(WHISPER_INITIAL_PROMPT)
    pattern = r'DEFAULT_INITIAL_PROMPT\s*=\s*\([\s\S]*?\)\n\n\ndef parse_args'
    replacement = f'DEFAULT_INITIAL_PROMPT = {prompt_literal}\n\n\ndef parse_args'
    patched, count = re.subn(pattern, replacement, src, count=1)
    if count != 1:
        raise RuntimeError("Không patch được DEFAULT_INITIAL_PROMPT trong stage_04_asr.py")
    asr_module_path.write_text(patched, encoding="utf-8")
    print("Whisper initial prompt: ON")
    print(WHISPER_INITIAL_PROMPT)
else:
    print("Whisper initial prompt: OFF")


## 12. Stage 04 - ASRMoE tiếng Việt

Output: `/kaggle/working/run_full/04_asr/transcript.json`.

Bản này chạy:
- Whisper large-v3 local trên GPU `WHISPER_DEVICE_INDEX`, dùng initial prompt nếu `USE_WHISPER_INITIAL_PROMPT=True`.
- PhoWhisper local/self-host trên GPU `VI_ASR_DEVICE_INDEX`, không dùng Hugging Face Inference API.
- ChunkFormer local trên GPU `VI_ASR_DEVICE_INDEX`.

Nếu máy/GPU không đủ RAM, giảm `AUDIO_LIMIT_SECONDS`, tắt bớt model, hoặc chuyển sang GPU mạnh hơn. Nhưng notebook này không fallback sang PhoWhisper API.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; tiếp tục nếu torch CUDA vẫn khả dụng.")

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
extra_ld_paths = [
    "/kaggle/working/cudnn8/nvidia/cudnn/lib",
    "/kaggle/working/cudnn8/nvidia/cublas/lib",
    "/kaggle/working/cudnn8/nvidia/cuda_nvrtc/lib",
]
extra_ld_paths = [p for p in extra_ld_paths if Path(p).exists()]
env["LD_LIBRARY_PATH"] = ":".join(extra_ld_paths + [env.get("LD_LIBRARY_PATH", "")]).rstrip(":")
print("extra LD paths:", extra_ld_paths)

# Hard-disable PhoWhisper API for this notebook. Model is downloaded and run locally.
env["PHOWHISPER_USE_HF_API"] = "0"
if "token" in globals() and token:
    env["HF_TOKEN"] = token
print("PhoWhisper mode: local/self-host")
print("PHOWHISPER_USE_HF_API:", env["PHOWHISPER_USE_HF_API"])
print("Whisper initial prompt:", "ON" if USE_WHISPER_INITIAL_PROMPT else "OFF")
print("Whisper hotwords:", WHISPER_HOTWORDS)
print("ASR context pad:", ASR_CONTEXT_PAD_BEFORE_SECONDS, ASR_CONTEXT_PAD_AFTER_SECONDS)

cmd = [
    "python", "stage_04_asr.py",
    "--segments_json", f"{OVERLAP_DIR}/segments.json",
    "--audio", f"{MUSIC_DIR}/cleaned_audio.wav",
    "--out", f"{ASR_DIR}/transcript.json",
    "--ASRMoE" if ASR_MOE else "--no-ASRMoE",
    "--no-whisperx_word_timestamps",
    "--initprompt" if USE_WHISPER_INITIAL_PROMPT else "--no-initprompt",
    "--whisper_arch", WHISPER_ARCH,
    "--compute_type", COMPUTE_TYPE,
    "--threads", str(ASR_THREADS),
    "--whisper_device_index", str(WHISPER_DEVICE_INDEX),
    "--vi_asr_device_index", str(VI_ASR_DEVICE_INDEX),
    "--asr_quality_guard" if ASR_QUALITY_GUARD else "--no-asr_quality_guard",
    "--asr_micro_segment_seconds", str(ASR_MICRO_SEGMENT_SECONDS),
    "--asr_short_segment_seconds", str(ASR_SHORT_SEGMENT_SECONDS),
    "--asr_vi_agreement_threshold", str(ASR_VI_AGREEMENT_THRESHOLD),
    "--asr_context_pad_before", str(ASR_CONTEXT_PAD_BEFORE_SECONDS),
    "--asr_context_pad_after", str(ASR_CONTEXT_PAD_AFTER_SECONDS),
    "--whisper_hotwords", WHISPER_HOTWORDS,
]
run_logged(cmd, "22_stage_04_asr.log", cwd="/kaggle/working/sommelier/podcast-pipeline", env=env, tail=30)


In [ ]:

with open(f"{ASR_DIR}/transcript.json", "r", encoding="utf-8") as f:
    transcript = json.load(f)

tr_segments = transcript["segments"]
print("Transcript segments:", len(tr_segments))
print("Metadata keys:", sorted(transcript.get("metadata", {}).keys()))

for s in tr_segments[:30]:
    start = float(s.get("start", 0))
    end = float(s.get("end", 0))
    speaker = s.get("speaker", "UNKNOWN")
    text = s.get("text", "").strip()
    print(f"[{start:07.2f} - {end:07.2f}] {speaker}: {text}")
 
if len(tr_segments) > 30:
    print(f"... hidden {len(tr_segments) - 30} more segments. Full JSON: {ASR_DIR}/transcript.json")


## 13. Stage 05 - Export final JSON và audio segment MP3

Output:
- `/kaggle/working/run_full/final/data_audio.json`
- `/kaggle/working/run_full/final/data_audio/clean_duplex_2speaker/*.mp3`
- `/kaggle/working/run_full/final/data_audio/overlap_review/*.mp3`
- `/kaggle/working/run_full/final/data_audio/exclude_or_extra_speaker/*.mp3`

In [ ]:

export_cmd = [
    "python", "stage_05_export.py",
    "--transcript_json", f"{ASR_DIR}/transcript.json",
    "--audio", f"{MUSIC_DIR}/cleaned_audio.wav",
    "--out_dir", FINAL_DIR,
    "--audio_name", "data_audio",
    "--expected_main_speakers", str(EXPECTED_MAIN_SPEAKERS),
]
if EXPORT_PARTITION_BY_DUPLEX_GROUP:
    export_cmd.append("--partition_by_duplex_group")
if MAIN_SPEAKERS:
    export_cmd.extend(["--main_speakers", ",".join(MAIN_SPEAKERS)])

run_logged(export_cmd, "23_stage_05_export.log", cwd="/kaggle/working/sommelier/podcast-pipeline", tail=25)

files = sorted(Path(RUN_DIR).glob("**/*"))
files = [p for p in files if p.is_file()]
print("Files:", len(files))
for p in files[:50]:
    print(p)
if len(files) > 50:
    print(f"... hidden {len(files) - 50} more files")


## 14. Review outputs by stage


In [ ]:
from pathlib import Path
import json
import pandas as pd
from pydub import AudioSegment
from IPython.display import Audio, display

STAGE_LOGS = {
    "0": ["00_prepare_audio_ffmpeg.log"],
    "audio": ["00_prepare_audio_ffmpeg.log"],
    "1": ["18_stage_01_diarize.log"],
    "01": ["18_stage_01_diarize.log"],
    "diarization": ["18_stage_01_diarize.log"],
    "2": ["19_stage_02_music_clean.log"],
    "02": ["19_stage_02_music_clean.log"],
    "music": ["19_stage_02_music_clean.log"],
    "3": ["20_stage_03_overlap_separate.log"],
    "03": ["20_stage_03_overlap_separate.log"],
    "overlap": ["20_stage_03_overlap_separate.log"],
    "4": ["22_stage_04_asr.log"],
    "04": ["22_stage_04_asr.log"],
    "asr": ["22_stage_04_asr.log"],
    "5": ["23_stage_05_export.log"],
    "05": ["23_stage_05_export.log"],
    "export": ["23_stage_05_export.log"],
    "6": ["24_stage_06_eval.log"],
    "06": ["24_stage_06_eval.log"],
    "eval": ["24_stage_06_eval.log"],
    "install": [
        "02_apt_update.log", "03_apt_install.log", "04_pip_base.log",
        "05_pip_requirements.log", "08_pip_nemo_asr.log", "10_pip_torch_stack.log",
        "12_pip_torchmetrics.log", "13_pip_numpy_numba.log", "14_pip_clearvoice.log", "21_pip_cudnn8.log",
    ],
}


def _stage_key(stage):
    return str(stage).lower().replace("stage", "").replace(" ", "").strip()


def show_stage_log(stage, lines=25):
    key = _stage_key(stage)
    logs = STAGE_LOGS.get(key, [])
    if not logs:
        print("Unknown stage. Try: 1, 2, 3, 4, 5, 6, install, audio")
        return

    for name in logs:
        path = Path(LOG_DIR_PATH) / name
        print("\n" + "=" * 90)
        print(path)
        print("=" * 90)
        if path.exists():
            print(tail_file(path, lines))
        else:
            print("Log chưa tồn tại. Hãy chạy stage tương ứng trước.")


def _load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Chưa có file: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _table(records, columns=None, n=5):
    df = pd.DataFrame(records[:n])
    if columns:
        columns = [c for c in columns if c in df.columns]
        df = df[columns]
    display(df)
    return df


def _slice_audio(audio_path, start, end, out_name, pad=0.15, max_seconds=45):
    audio_path = Path(audio_path)
    if not audio_path.exists():
        print("Missing audio:", audio_path)
        return None

    start = max(0.0, float(start) - pad)
    end = max(start, float(end) + pad)
    if end - start > max_seconds:
        end = start + max_seconds

    audio = AudioSegment.from_file(audio_path)
    out = Path(PREVIEW_DIR) / out_name
    out.parent.mkdir(parents=True, exist_ok=True)
    audio[int(start * 1000):int(end * 1000)].export(out, format="wav")
    return out


def _show_segment_audio(segments, audio_path, n=5, pad=0.15, prefer_enhanced=True):
    for i, seg in enumerate(segments[:n]):
        print("\n--- audio", i, "---")
        print(f"[{float(seg.get('start', 0)):.2f} - {float(seg.get('end', 0)):.2f}]", seg.get("speaker", ""))

        enhanced = seg.get("enhanced_audio_path") if prefer_enhanced else None
        if enhanced and Path(enhanced).exists():
            print("enhanced:", enhanced)
            display(Audio(enhanced))
            continue

        out = _slice_audio(
            audio_path,
            seg.get("start", 0),
            seg.get("end", 0),
            f"_review_stage_audio_{i:03d}.wav",
            pad=pad,
        )
        if out:
            display(Audio(str(out)))


def review_stage(stage, n=5, log_lines=25, play_audio=True, pad=0.15):
    """Review compact một stage: log ngắn + 5 dòng đầu + tối đa 5 audio clip."""
    key = _stage_key(stage)
    print("Review stage:", stage)
    show_stage_log(key, lines=log_lines)

    if key in {"1", "01", "diarization"}:
        data = _load_json(Path(DIAR_DIR) / "diarization.json")
        segments = data.get("segments", [])
        print("\nDiarization segments:", len(segments))
        _table(segments, ["index", "start", "end", "speaker"], n=n)
        if play_audio:
            _show_segment_audio(segments, AUDIO_WAV, n=n, pad=pad, prefer_enhanced=False)
        return

    if key in {"2", "02", "music"}:
        data = _load_json(Path(MUSIC_DIR) / "segment_flags.json")
        segments = data.get("segments", [])
        flags = data.get("segment_demucs_flags", [])
        rows = []
        for i, seg in enumerate(segments):
            row = dict(seg)
            row["demucs_flag"] = bool(flags[i]) if i < len(flags) else False
            rows.append(row)
        print("\nMusic-clean segments:", len(rows))
        print("Demucs flagged:", sum(bool(x) for x in flags), "/", len(flags))
        _table(rows, ["index", "start", "end", "speaker", "demucs_flag"], n=n)
        if play_audio:
            _show_segment_audio(rows, Path(MUSIC_DIR) / "cleaned_audio.wav", n=n, pad=pad, prefer_enhanced=False)
        return

    if key in {"3", "03", "overlap"}:
        data = _load_json(Path(OVERLAP_DIR) / "segments.json")
        segments = data.get("segments", [])
        print("\nStage 03 segments:", len(segments))
        _table(segments, ["index", "start", "end", "speaker", "has_overlap", "overlap_count", "overlap_total_seconds", "needs_manual_review", "train_quality_label", "is_separated", "separation_method", "separation_status", "low_confidence_overlap_count", "duplex_train_group", "duplex_group_reason", "enhanced_audio_path"], n=n)
        if play_audio:
            _show_segment_audio(segments, Path(MUSIC_DIR) / "cleaned_audio.wav", n=n, pad=pad, prefer_enhanced=True)
        return

    if key in {"4", "04", "asr"}:
        data = _load_json(Path(ASR_DIR) / "transcript.json")
        segments = data.get("segments", [])
        print("\nTranscript segments:", len(segments))
        _table(
            segments,
            ["index", "start", "end", "speaker", "text", "asr_quality_source", "asr_quality_actions", "text_whisper", "text_phowhisper", "text_chunkformer"],
            n=n,
        )
        if play_audio:
            _show_segment_audio(segments, Path(MUSIC_DIR) / "cleaned_audio.wav", n=n, pad=pad, prefer_enhanced=True)
        return

    if key in {"5", "05", "export"}:
        final_dir = Path(FINAL_DIR) / "data_audio"
        files = sorted(final_dir.glob("*.mp3"))
        print("\nExported MP3:", len(files))
        display(pd.DataFrame({"i": range(min(n, len(files))), "path": [str(p) for p in files[:n]]}))
        if play_audio:
            for i, path in enumerate(files[:n]):
                print("\n--- exported", i, "---")
                print(path)
                display(Audio(str(path)))
        return

    print("Stage này chỉ có log. Dùng show_stage_log(stage) để xem log.")


print("Dùng: review_stage(1), review_stage(2), review_stage(3), review_stage(4), review_stage(5), review_stage(6)")
print("Ví dụ chỉ xem log stage 4: show_stage_log(4, lines=80)")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
# review_stage(1)
review_stage(4, n=10, play_audio=True)


## 15. Stage 06 - Eval metrics

Tinh metric sau khi chay xong cac stage: audio info, segment stats, ASR suspicious segments, export consistency va recommendation.

In [ ]:

from pathlib import Path
from IPython.display import Markdown, display

run_logged([
    "python", "stage_06_eval.py",
    "--run_dir", RUN_DIR,
    "--print_markdown",
], "24_stage_06_eval.log", cwd="/kaggle/working/sommelier/podcast-pipeline", tail=80)

report_md = Path(EVAL_DIR) / "eval_report.md"
report_json = Path(EVAL_DIR) / "eval_report.json"
print("Eval markdown:", report_md)
print("Eval json:", report_json)

if report_md.exists():
    display(Markdown(report_md.read_text(encoding="utf-8")))
else:
    print("Eval report chua ton tai. Kiem tra log stage 06.")


In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

run_dir = Path(RUN_DIR)
zip_base = Path("/kaggle/working/run_full_download")

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(run_dir.parent),
    base_dir=run_dir.name,
)

print("Created:", zip_path)
display(FileLink(zip_path))